# S5.M4-1 mock 重合 80 上 M6 的真实隐语 SGB 复核

**目的**：S5.M2 推迟了 mock 重合 80 上 M6（6 模型 bagging）的真实 SGB 复核。代理上 M6 − 优化版为 +0.009（S5.M1，6 seed），交叉验证 R1 在该任务上也选了 P+M6（S5.M3）。本步检验：
- M6 − 优化版在 SGB 上是否成立；
- 同配置下 SGB 与代理的差是否在 0.005 以内（[DR-M5-013](../../../registry/decision_records/DR-M5-013.yaml) · D-5，沿用 DR-M5-012 · D-1）。

**运行方式**：
- 仓库组件 `method_screen_sgb`（与 S5.M2 同一实现），隐语 secretflow-lite 1.13.0b0，debug 单进程两方模拟，OU；
- M6 为 6 个 SGB（行采样 0.8，成员 seed 1100–1105）的概率平均；树数、深度、分桶沿用 S5.M1 优化版，不再调参；
- 优化版沿用 S5.M2 已拟合的打分，不重跑；两者在同一测试集上配对。

**数据**：公开数据，**开发证据**，不代表汇丰效果。逐行打分只在私有目录。

In [1]:
import hashlib
import json
import logging
import os
import pathlib
import subprocess
import sys
import tempfile

import numpy as np
import pandas as pd
import yaml

HASH_PREFIX_LEN = 12
logging.getLogger().setLevel(logging.ERROR)        # 隐语运行日志不进 notebook
REPO_ROOT = pathlib.Path.cwd()
while not (REPO_ROOT / "AGENTS.md").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
CONFIG_PATH = REPO_ROOT / "modules/m5_modeling/configs/s5_m4_sgb_m6_mock80.yaml"
m4 = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
BASE_PATH = REPO_ROOT / m4["base_config"]
cfg = yaml.safe_load(BASE_PATH.read_text(encoding="utf-8"))
COMPONENTS = [REPO_ROOT / "modules/m5_modeling/components" / n
              for n in ("method_screen.py", "method_screen_data.py", "method_screen_sgb.py")]


def sha(path, n=HASH_PREFIX_LEN):
    return hashlib.sha256(pathlib.Path(path).read_bytes()).hexdigest()[:n]


git_sha = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True, cwd=REPO_ROOT).strip()
dirty = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True,
                       cwd=REPO_ROOT).stdout.strip() != ""
print("config:", CONFIG_PATH.relative_to(REPO_ROOT), "sha256:" + sha(CONFIG_PATH))
print("base config:", BASE_PATH.relative_to(REPO_ROOT), "sha256:" + sha(BASE_PATH))
print("决策记录:", m4["decision_record"], "| step:", m4["step_id"], "|", m4["purpose"])
print("seed:", m4["seed"], "| seeds:", m4["seeds"])
print("git:", git_sha + ("（工作区含本步未提交改动）" if dirty else ""))
for c in COMPONENTS:
    print("组件:", c.relative_to(REPO_ROOT), "sha256:" + sha(c))
PRIV = pathlib.Path(os.environ[cfg["private_root_env"]])
RUN = pathlib.Path(os.environ[cfg["run_dir_env"]])
SGB = pathlib.Path(os.environ[m4["sgb_dir_env"]])
launch = {}
print("运行时指纹（会话驱动脚本逐段记录；路径只留文件名）：")
for line in (SGB / m4["runtime_fingerprint"]).read_text(encoding="utf-8").splitlines():
    parts = line.split()
    if len(parts) == 2 and parts[1].endswith(".py"):
        launch[pathlib.Path(parts[1]).name] = parts[0]
        print("  sha256:" + parts[0][:HASH_PREFIX_LEN], pathlib.Path(parts[1]).name)
    else:
        print(" ", line)
for c in COMPONENTS:
    if c.name in launch:
        assert launch[c.name] == hashlib.sha256(c.read_bytes()).hexdigest(), c.name
print("运行时使用的仓库组件与当前仓库逐字节相同：", sorted(n for n in launch if n != "sgb_m6_mock80.py"))

config: modules/m5_modeling/configs/s5_m4_sgb_m6_mock80.yaml sha256:4539172abeb3
base config: modules/m5_modeling/configs/s5_m1_method_screen.yaml sha256:761bae66da90
决策记录: registry/decision_records/DR-M5-013.yaml | step: S5.M4 | mock 重合 80 上 M6 的真实 SGB 复核——M6 − 优化版在 SGB 上是否成立，SGB 与同配置代理的差是否在 0.005 以内
seed: 11 | seeds: [11, 22, 33, 44, 55, 66]
git: 674a299（工作区含本步未提交改动）
组件: modules/m5_modeling/components/method_screen.py sha256:05e19e025d0b
组件: modules/m5_modeling/components/method_screen_data.py sha256:e4bfdf6fe3d4
组件: modules/m5_modeling/components/method_screen_sgb.py sha256:2e54c618305f
运行时指纹（会话驱动脚本逐段记录；路径只留文件名）：
  sha256:9c56c2f0d7a4 sgb_m6_mock80.py
  sha256:2e54c618305f method_screen_sgb.py
  sha256:05e19e025d0b method_screen.py
  2026-11-03T14:50:41+08:00
  --- 2026-11-03T16:53:01+08:00 第 2 段（第 1 段 7000 秒到时，成员 0 已存档）
  --- 2026-11-03T18:50:05+08:00 第 3 段（第 2 段 7000 秒到时，成员 0–2 已存档；本段上限改为 7150 秒）
  --- 2026-11-03T20:38:58+08:00 第 4 段（第 3 段在成员 4 存档后手动停止，未等到时；成员 0–4 已存档）
运行时使用的仓库组件与

## 1. 运行来源与成员核对

运行由会话驱动脚本直接调用仓库组件 `method_screen_sgb.confirm`，在后台分段执行（每段约 2 小时上限），成员逐个存档，到时后续跑；各段启动时间与代码 sha256 见上一格。

本 notebook 不重新拟合（每个成员约 1 小时），改为核对：
- 运行时使用的仓库组件与当前仓库逐字节相同（上一格断言）；
- 6 个成员的 seed 与组件定义相同，M6 存档打分等于 6 个成员打分的平均（下一格断言）；
- 每个成员另训同 seed、同配置的代理，给出成员级的 SGB − 代理。

In [2]:
from modules.m5_modeling.components import method_screen as MS
from modules.m5_modeling.components import method_screen_sgb as SG

PRED_MATCH = 1e-6       # 打分以 float32 存档：M6 存档与成员平均允许的差
SECONDS_PER_HOUR = 3600
P = MS.Params(cfg)
NAME = m4["dataset"]
RES = json.loads((RUN / (NAME + ".json")).read_text(encoding="utf-8"))
sc = MS.Screen(NAME, cfg, PRIV, pathlib.Path(tempfile.mkdtemp()) / "o")
fv = sc.v["final"]
ye = fv["ev_y"]
mcfg, bins, _, b_proxy, _, _ = SG.inputs(P, sc, RES, "F_M6")
rows, sgb_m, proxy_m = [], [], []
for j, sd in enumerate(SG.bag_seeds(P)):
    meta = json.loads((SGB / ("%s__F_M6_m%d.json" % (NAME, j))).read_text(encoding="utf-8"))
    assert meta["seed"] == sd
    pj = np.load(SGB / ("%s__F_M6_m%d.npy" % (NAME, j))).astype(float)
    qj = MS.train_pred(P, fv["fit_xc"], fv["fit_y"], fv["ev_xc"], mcfg, bins, b_proxy, sd)
    sgb_m.append(pj)
    proxy_m.append(qj)
    rows.append({"member": j, "seed": sd, "fit_seconds": meta["fit_seconds"],
                 "sgb_auroc": MS.auc(ye, pj), "proxy_auroc": MS.auc(ye, qj)})
agg = np.load(SGB / (NAME + "__F_M6.npy")).astype(float)
gap = float(np.max(np.abs(agg - np.mean(sgb_m, axis=0))))
assert gap <= PRED_MATCH
members = pd.DataFrame(rows)
members["sgb_minus_proxy"] = members["sgb_auroc"] - members["proxy_auroc"]
print("配置：%d 棵、深度 %d、%d 桶、行采样 %.1f；测试行数 %d" % (mcfg["rounds"], mcfg["depth"], bins, mcfg["sub"], len(ye)))
print("M6 存档打分与 6 个成员平均的最大差：%.2g" % gap)
print(members.round(cfg["round_auc"]).to_string(index=False))
print("合计拟合 %.0f 秒（约 %.1f 小时）" % (members["fit_seconds"].sum(), members["fit_seconds"].sum() / SECONDS_PER_HOUR))

配置：83 棵、深度 6、33 桶、行采样 0.8；测试行数 17391
M6 存档打分与 6 个成员平均的最大差：3e-08
 member  seed  fit_seconds  sgb_auroc  proxy_auroc  sgb_minus_proxy
      0  1100       5283.8     0.9074       0.9078          -0.0004
      1  1101       3462.1     0.9085       0.9084           0.0001
      2  1102       3438.3     0.9091       0.9077           0.0014
      3  1103       3079.5     0.9058       0.9065          -0.0007
      4  1104       3388.9     0.9100       0.9051           0.0049
      5  1105       5080.3     0.9045       0.9050          -0.0006
合计拟合 23733 秒（约 6.6 小时）


## 2. 代理与 SGB 的一致性，以及 SGB 上 M6 的增益

- 优化版一行与 S5.M2 公开汇总相同（断言），说明配对的打分与 bootstrap 抽样一致；
- 区间只含测试集抽样（客户簇 bootstrap），不含训练随机性；
- S5.M1 的代理对比是 6 个 seed 的结果，作对照列。

In [3]:
AUROC_MATCH = 1e-4      # 打分以 float32 存档，与运行时 float64 打分算出的 AUROC 允许的差
CSV_MATCH = 1e-5        # S5.M2 公开汇总按 5 位小数存
OUT = REPO_ROOT / m4["results_dir"]
OUT.mkdir(parents=True, exist_ok=True)
tol = m4["proxy_tolerance"]
done = json.loads((SGB / (NAME + ".json")).read_text(encoding="utf-8"))
sgb = {m: np.load(SGB / ("%s__%s.npy" % (NAME, m))).astype(float) for m in m4["methods"]}
proxy = {"F_opt_p": SG.proxy_pred(P, sc, RES, "F_opt_p"), "F_M6": np.mean(proxy_m, axis=0)}
ev = SG.evaluate(P, sc, RES, sgb, proxy, [tuple(x) for x in m4["pairs"]])
meth = []
for m, v in ev["methods"].items():
    assert abs(v["sgb_auroc"] - done[m]["sgb_auroc"]) < AUROC_MATCH
    assert abs(v["proxy_auroc"] - done[m]["proxy_seed11_auroc"]) < AUROC_MATCH
    d = v["sgb_minus_proxy"]
    meth.append({"dataset": NAME, "method": m, "trees": done[m]["trees"], "depth": done[m]["depth"], "bins": done[m]["bins"],
                 "sgb_auroc": v["sgb_auroc"], "proxy_auroc": v["proxy_auroc"], "proxy_6seed_auroc": done[m]["proxy_6seed_auroc"],
                 "sgb_minus_proxy": d["diff"], "ci_low": d["ci_low"], "ci_high": d["ci_high"],
                 "within_tolerance": abs(d["diff"]) <= tol, "fit_seconds": done[m]["fit_seconds"]})
meth = pd.DataFrame(meth)
s5m2 = pd.read_csv(REPO_ROOT / m4["s5m2_results"] / "sgb_vs_proxy.csv")
old = s5m2[(s5m2["dataset"] == NAME) & (s5m2["method"] == "F_opt_p")].iloc[0]
new = meth[meth["method"] == "F_opt_p"].iloc[0]
for k in ("sgb_auroc", "proxy_auroc", "sgb_minus_proxy", "ci_low", "ci_high"):
    assert abs(float(old[k]) - float(new[k])) < CSV_MATCH, k
print("优化版一行与 S5.M2 公开汇总相同（同一批 bootstrap 抽样）。")
con = []
for label, c in ev["contrasts"].items():
    s, q = c["sgb"], c["proxy_6seed"]
    con.append({"dataset": NAME, "contrast": label, "sgb_diff": s["diff"], "sgb_ci_low": s["ci_low"],
                "sgb_ci_high": s["ci_high"], "sgb_p": s["p"], "proxy_6seed_diff": q["diff"],
                "proxy_6seed_ci_low": q["ci_low"], "proxy_6seed_ci_high": q["ci_high"]})
con = pd.DataFrame(con)
meth.to_csv(OUT / "sgb_vs_proxy.csv", index=False, float_format="%.5f")
con.to_csv(OUT / "sgb_contrasts.csv", index=False, float_format="%.5f")
members.to_csv(OUT / "sgb_members.csv", index=False, float_format="%.5f")
print("同配置下 SGB 与代理（M6 为同样 6 个 seed 的平均）：")
print(meth.round(cfg["round_auc"]).to_string(index=False))
print("SGB 上的 M6 − 优化版（单次拟合，区间只含测试集抽样）与 S5.M1 代理（6 seed）：")
print(con.round(cfg["round_auc"]).to_string(index=False))
m6 = meth[meth["method"] == "F_M6"].iloc[0]
print("DR-M5-013 · D-5 推翻条件（|SGB − 代理| > %.3f）：%s" % (tol, "未触发" if m6["within_tolerance"] else "触发，M6 的代理结论以 SGB 为准"))

优化版一行与 S5.M2 公开汇总相同（同一批 bootstrap 抽样）。
同配置下 SGB 与代理（M6 为同样 6 个 seed 的平均）：
       dataset  method  trees  depth  bins  sgb_auroc  proxy_auroc  proxy_6seed_auroc  sgb_minus_proxy  ci_low  ci_high  within_tolerance  fit_seconds
mock_overlap80 F_opt_p     83      6    33     0.9106       0.9044             0.9044           0.0062  0.0010   0.0116             False       3094.1
mock_overlap80    F_M6     83      6    33     0.9142       0.9128             0.9136           0.0013 -0.0019   0.0042              True      23732.9
SGB 上的 M6 − 优化版（单次拟合，区间只含测试集抽样）与 S5.M1 代理（6 seed）：
       dataset       contrast  sgb_diff  sgb_ci_low  sgb_ci_high  sgb_p  proxy_6seed_diff  proxy_6seed_ci_low  proxy_6seed_ci_high
mock_overlap80 F_M6 − F_opt_p    0.0036      0.0001       0.0068  0.038            0.0092              0.0059               0.0128
DR-M5-013 · D-5 推翻条件（|SGB − 代理| > 0.005）：未触发


## 3. 读法

**来源核对**：运行时使用的仓库组件与当前仓库逐字节相同；6 个成员 seed 为 1100–1105，M6 存档打分等于成员平均（最大差 3e-08，float32 末位）。

**代理与 SGB 是否一致**（[DR-M5-013](../../../registry/decision_records/DR-M5-013.yaml) · D-5）：
- M6：SGB 0.9142，同配置代理 0.9128，差 +0.0013 [−0.0019, +0.0042]，在 0.005 以内，推翻条件未触发。
- 成员级的差在 −0.0007 ~ +0.0049 之间。
- 优化版一行与 S5.M2 相同（SGB 比代理高 0.0062，S5.M2 已按 D-1 降级）。

**SGB 上 M6 是否成立**（单次拟合；区间只含测试集抽样）：
- M6 − 优化版 = **+0.0036 [+0.0001, +0.0068]**，p = 0.038，区间刚好不含 0；
- 代理（S5.M1，6 seed）为 +0.0092 [+0.0059, +0.0128]。SGB 上幅度约为代理的 4 成。
- 原因在优化版一侧：SGB 优化版本身比代理高 0.006（S5.M2），M6 在两种引擎上几乎相同，所以差距被压缩。
- 6 个行采样 0.8 的成员单独看都不高于不采样的优化版（0.9045 ~ 0.9100 对 0.9106），平均后到 0.9142：增益来自平均，不来自单个成员。

**代价**：6 个成员合计拟合 23,733 秒（约 6.6 小时），优化版单模型 3,094 秒，实测约 7.7 倍（其中两段与其他任务争用 CPU；无争用时每个成员约 3,450 秒，约 6.7 倍）。换来的增益是 +0.004。

**合并 S5.M2**：M6 在真实 SGB 上做过 3 个任务——FedAds +0.011（成立）、mock 重合 80 +0.004（刚好成立）、COIL2000 +0.003（不显著）。方向一致，幅度都在 +0.003 ~ +0.011。

**边界**：公开数据开发证据，不代表汇丰效果；单次拟合不含训练随机性（每个成员各一个 seed）；debug 模拟下打分直接取回，只用于离线评估。技术侧自评，未经法务复核（W-001）。